# 06 - Controlled Model Comparison and Measured Ablations

Combines baseline/custom metrics and runs measured context-length ablations (64, 128, 256) under a consistent training budget.

**Context length:** The **Transformer** sweep matches the architecture in notebook 04 (`TransformerNextToken`, same hparams) and is the primary ablation for the final demo model. A small **GRU** sweep is kept as an optional lightweight baseline for comparison only.


In [17]:
# Optional for Colab
# !pip install -q torch pandas numpy


In [18]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

PROJECT_OUTPUT_DIR = Path('/content/project_outputs')
CACHE_DIR = PROJECT_OUTPUT_DIR / 'cache'
TABLE_DIR = PROJECT_OUTPUT_DIR / 'tables'

# Inputs from prior notebooks (edit paths if you moved outputs).
BASELINE_CSV = TABLE_DIR / '03_baseline_results.csv'
GRU_METRICS_CSV = TABLE_DIR / '04_training_metrics.csv'
TRANSFORMER_METRICS_GLOB = '04_training_metrics_transformer_*.csv'


def _read_csv_optional(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f'[warn] missing CSV: {path}')
        return pd.DataFrame()
    return pd.read_csv(path)


baseline_df = _read_csv_optional(BASELINE_CSV)
gru_df = _read_csv_optional(GRU_METRICS_CSV)
_tf_paths = sorted(TABLE_DIR.glob(TRANSFORMER_METRICS_GLOB))
if _tf_paths:
    tf_df = pd.concat([pd.read_csv(p) for p in _tf_paths], ignore_index=True)
    print('Loaded transformer metrics:', [p.name for p in _tf_paths])
else:
    tf_df = pd.DataFrame()
    print(f'[warn] no files matching {TRANSFORMER_METRICS_GLOB} under {TABLE_DIR}')

with open('/content/sequence_cache_with_quant_ids.json', 'r') as f:
    cache = json.load(f)

V = len(cache['vocab'])
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Aliases for older comparison snippets
base = baseline_df
train = gru_df


Loaded transformer metrics: ['04_training_metrics_transformer_quantized_time.csv']


In [19]:
# --- Baselines (03) + GRU (04) + Transformer (04 optional): long table ---
CORE = ['setting', 'model', 'accuracy', 'cross_entropy', 'perplexity', 'top5']


def _subset_core(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    cols = [c for c in CORE if c in df.columns]
    return df[cols].copy()


model_comparison_long = pd.concat(
    [_subset_core(baseline_df), _subset_core(gru_df), _subset_core(tf_df)],
    ignore_index=True,
)
if model_comparison_long.empty:
    raise FileNotFoundError('No metrics loaded; check BASELINE_CSV / GRU_METRICS_CSV paths and TABLE_DIR.')

model_comparison_long = model_comparison_long.sort_values(
    ['setting', 'perplexity'], ascending=[True, True]
).reset_index(drop=True)
display(model_comparison_long)

# --- Per-setting summary: best baseline vs GRU vs Transformer (perplexity) ---
BASELINE_MODELS = {'unigram', 'bigram_markov'}


def _best_baseline(sub_base: pd.DataFrame):
    if sub_base.empty:
        return np.nan, ''
    i = sub_base['perplexity'].idxmin()
    return float(sub_base.loc[i, 'perplexity']), str(sub_base.loc[i, 'model'])


summary_rows = []
for setting in sorted(model_comparison_long['setting'].unique()):
    sub = model_comparison_long[model_comparison_long['setting'] == setting]
    sub_b = sub[sub['model'].isin(BASELINE_MODELS)]
    bb_ppl, bb_name = _best_baseline(sub_b)
    gru_sub = sub[sub['model'] == 'gru_custom']
    tf_sub = sub[sub['model'] == 'transformer_custom']
    gru_ppl = float(gru_sub['perplexity'].iloc[0]) if len(gru_sub) else np.nan
    tf_ppl = float(tf_sub['perplexity'].iloc[0]) if len(tf_sub) else np.nan

    def _gain(bb, nn):
        if bb == bb and nn == nn:
            return float(bb - nn)
        return np.nan

    summary_rows.append({
        'setting': setting,
        'best_baseline_model': bb_name,
        'best_baseline_ppl': bb_ppl,
        'gru_ppl': gru_ppl,
        'transformer_ppl': tf_ppl,
        'ppl_gain_vs_best_baseline_gru': _gain(bb_ppl, gru_ppl),
        'ppl_gain_vs_best_baseline_transformer': _gain(bb_ppl, tf_ppl),
    })

comparison_df = pd.DataFrame(summary_rows)
display(comparison_df)


,setting,model,accuracy,cross_entropy,perplexity,top5
0,quantized_time,transformer_custom,0.285278,2.738850,15.469179,0.644640
1,quantized_time,gru_custom,0.280280,2.743658,15.543741,0.638600
2,quantized_time,bigram_markov,0.133615,4.421279,83.202651,0.347258
3,quantized_time,unigram,0.125658,4.448440,85.493451,0.312398
4,raw_time,gru_custom,0.187228,3.617372,37.239575,0.432841
5,raw_time,unigram,0.025639,5.389597,219.115069,0.113485
6,raw_time,bigram_markov,0.033507,6.254606,520.404251,0.146702


,setting,best_baseline_model,best_baseline_ppl,gru_ppl,transformer_ppl,ppl_gain_vs_best_baseline_gru,ppl_gain_vs_best_baseline_transformer
0,quantized_time,bigram_markov,83.202651,15.543741,15.469179,67.658909,67.733472
1,raw_time,unigram,219.115069,37.239575,NaN,181.875494,NaN


## Measured ablation: context length (64, 128, 256)

Windows are rebuilt from **file-level split-safe** `quant_tokens` sequences (same pipeline as before). **`ctx_results_gru`**: optional small GRU baseline (4 epochs, cheap probe). **`ctx_results_tf`**: primary ablation — **Transformer** from notebook 04 (`emb_dim=192`, `nhead=6`, `num_layers=3`, AdamW + cosine, **5 epochs** per context length for a fair apples-to-apples sweep). Batch size is **64** for 64/128 and **32** for 256 to reduce GPU OOM risk without changing `seq_len`.

In [20]:
# Measured ablation: Transformer context length (64, 128, 256).
# Same underlying quant_ids per file/split; each seq_len defines different (prefix -> next token) pairs.

file_records = cache['file_sequences']


def build_windows(ids, seq_len=128, stride=64):
    X, y = [], []
    for i in range(0, max(0, len(ids) - seq_len), stride):
        j = i + seq_len
        if j < len(ids):
            X.append(ids[i:j])
            y.append(ids[j])
    return X, y


def flatten_split(file_records, split_name, seq_len, stride):
    """Build windows from notebook 02 cache.

    Preferred path uses per-file `quant_ids`. Older caches only store `quant_X`/`quant_y`.
    """
    X, y = [], []
    cache_seq_len = int(cache.get('seq_len', 128))

    for rec in file_records:
        if rec['split_norm'] != split_name:
            continue

        if 'quant_ids' in rec:
            ids = list(rec['quant_ids'])
            xw, yw = build_windows(ids, seq_len=seq_len, stride=stride)
        elif 'quant_X' in rec and 'quant_y' in rec:
            # Legacy cache: only supports the seq_len used when notebook 02 built windows.
            if seq_len != cache_seq_len:
                raise RuntimeError(
                    f"Cache has no quant_ids and only prebuilt quant_X/quant_y at seq_len={cache_seq_len}; "
                    f"cannot rebuild seq_len={seq_len}. Re-run notebook 02 to save quant_ids."
                )
            xw, yw = rec['quant_X'], rec['quant_y']
        else:
            raise KeyError("Expected one of ['quant_ids'] or ['quant_X','quant_y'] in cache file_sequences records.")

        X.extend(xw)
        y.extend(yw)
    return X, y


def to_loader(X, y, batch_size=64, shuffle=True):
    X_t = torch.tensor(np.array(X), dtype=torch.long)
    y_t = torch.tensor(np.array(y), dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)


def test_metrics(model, te_loader):
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for xb, yb in te_loader:
            xb = xb.to(device)
            probs = torch.softmax(model(xb), dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_y.extend(yb.numpy().tolist())
    probs = np.vstack(all_probs)
    preds = probs.argmax(axis=1)
    acc = float(np.mean(np.array(all_y) == preds))
    ptrue = np.array([max(probs[i, t], 1e-12) for i, t in enumerate(all_y)])
    ce = float(-np.mean(np.log(ptrue)))
    ppl = float(np.exp(ce))
    k = min(5, probs.shape[1])
    topk = float(np.mean([all_y[i] in np.argpartition(probs[i], -k)[-k:] for i in range(len(all_y))]))
    return acc, ce, ppl, topk


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]


class TransformerNextToken(nn.Module):
    def __init__(self, vocab_size, emb_dim=192, nhead=6, num_layers=3, ff_mult=4, dropout=0.2, max_len=512):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.pos = PositionalEncoding(emb_dim, max_len=max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=emb_dim * ff_mult,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        h = self.emb(x)
        h = self.pos(h)
        h = self.enc(h)
        h = self.drop(h[:, -1, :])
        return self.fc(h)


# Match notebook 04 Transformer; fixed train budget per seq_len for fair comparison.
TF_ABL_EPOCHS = 5
TF_EMB = 192
TF_HEADS = 6
TF_LAYERS = 3
TF_FF_MULT = 4
TF_DROPOUT = 0.2
TF_LR = 1e-3
# Cap windows for runtime (same cap for every seq_len).
ABL_MAX_TRAIN_WINDOWS = 10000


def eval_context_len_transformer(seq_len):
    torch.manual_seed(SEED + int(seq_len))
    stride = max(16, seq_len // 2)

    trX, trY = flatten_split(file_records, 'train', seq_len=seq_len, stride=stride)
    vaX, vaY = flatten_split(file_records, 'val', seq_len=seq_len, stride=stride)
    teX, teY = flatten_split(file_records, 'test', seq_len=seq_len, stride=stride)

    trX, trY = trX[:ABL_MAX_TRAIN_WINDOWS], trY[:ABL_MAX_TRAIN_WINDOWS]
    vaX, vaY = vaX[: ABL_MAX_TRAIN_WINDOWS // 2], vaY[: ABL_MAX_TRAIN_WINDOWS // 2]
    teX, teY = teX[: ABL_MAX_TRAIN_WINDOWS // 2], teY[: ABL_MAX_TRAIN_WINDOWS // 2]

    if len(trX) == 0 or len(teX) == 0:
        raise RuntimeError(f'No windows for seq_len={seq_len}; check cache / seq_len.')

    bs = 32 if seq_len >= 256 else 64
    tr = to_loader(trX, trY, batch_size=bs, shuffle=True)
    te = to_loader(teX, teY, batch_size=bs, shuffle=False)

    # max_len must be >= longest sequence fed to the model (here == seq_len).
    model = TransformerNextToken(
        vocab_size=V,
        emb_dim=TF_EMB,
        nhead=TF_HEADS,
        num_layers=TF_LAYERS,
        ff_mult=TF_FF_MULT,
        dropout=TF_DROPOUT,
        max_len=max(512, seq_len),
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=TF_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, TF_ABL_EPOCHS))
    crit = nn.CrossEntropyLoss()

    for _ in range(TF_ABL_EPOCHS):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()

    acc, ce, ppl, topk = test_metrics(model, te)
    return {
        'setting': 'quantized_time',
        'architecture': 'transformer_custom',
        'context_len': seq_len,
        'stride': stride,
        'n_train_windows': len(trX),
        'n_val_windows': len(vaX),
        'n_test_windows': len(teX),
        'accuracy': acc,
        'cross_entropy': ce,
        'perplexity': ppl,
        'top5': topk,
        'emb_dim': TF_EMB,
        'n_heads': TF_HEADS,
        'n_layers': TF_LAYERS,
        'epochs': TF_ABL_EPOCHS,
        'batch_size': bs,
        'lr': TF_LR,
        'dropout': TF_DROPOUT,
    }


default_contexts = [64, 128, 256]
if file_records and ('quant_ids' not in file_records[0]) and ('quant_X' in file_records[0]):
    # Legacy cache from notebook 02 without per-file token id streams.
    contexts = [int(cache.get('seq_len', 128))]
    print(f"[warn] legacy cache detected (no quant_ids); running only context_len={contexts[0]}.")
else:
    contexts = default_contexts

ctx_results_transformer = pd.DataFrame([eval_context_len_transformer(c) for c in contexts])
display(ctx_results_transformer)


/tmp/ipykernel_780/3328197290.py:102: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


,setting,architecture,context_len,stride,n_train_windows,n_val_windows,n_test_windows,accuracy,cross_entropy,perplexity,top5,emb_dim,n_heads,n_layers,epochs,batch_size,lr,dropout
0,quantized_time,transformer_custom,64,32,10000,5000,5000,0.1854,3.705726,40.679557,0.4804,192,6,3,5,64,0.001,0.2
1,quantized_time,transformer_custom,128,64,10000,5000,5000,0.1772,3.648463,38.415558,0.4778,192,6,3,5,64,0.001,0.2
2,quantized_time,transformer_custom,256,128,10000,5000,5000,0.2034,3.641457,38.147362,0.5064,192,6,3,5,32,0.001,0.2


In [21]:
model_comparison_long.to_csv(TABLE_DIR / '06_model_comparison.csv', index=False)
comparison_df.to_csv(TABLE_DIR / '06_ablation_preprocessing.csv', index=False)
ctx_results_transformer.to_csv(TABLE_DIR / '06_ablation_context_length_transformer.csv', index=False)
print(
    'Saved notebook 06 outputs:',
    TABLE_DIR / '06_model_comparison.csv',
    TABLE_DIR / '06_ablation_preprocessing.csv',
    TABLE_DIR / '06_ablation_context_length_transformer.csv',
)


Saved notebook 06 outputs: /content/project_outputs/tables/06_model_comparison.csv /content/project_outputs/tables/06_ablation_preprocessing.csv /content/project_outputs/tables/06_ablation_context_length_transformer.csv
